In [1]:
from pathlib import Path

root = Path(r"C:/potenup3/transdeep/download (1)/15.인도보행영상/뎁스프리딕션")

def print_tree(path: Path, max_depth=3, prefix=""):
    if max_depth < 0:
        return
    items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    for i, p in enumerate(items):
        connector = "└── " if i == len(items)-1 else "├── "
        print(prefix + connector + p.name)
        if p.is_dir():
            extension = "    " if i == len(items)-1 else "│   "
            print_tree(p, max_depth-1, prefix + extension)

print(root)
print_tree(root, max_depth=2)  # 깊이 조절

C:\potenup3\transdeep\download (1)\15.인도보행영상\뎁스프리딕션
├── Depth_001~005.zip.part0
├── Depth_001~005.zip.part1073741824
├── Depth_001~005.zip.part10737418240
├── Depth_001~005.zip.part11811160064
├── Depth_001~005.zip.part12884901888
├── Depth_001~005.zip.part13958643712
├── Depth_001~005.zip.part15032385536
├── Depth_001~005.zip.part16106127360
├── Depth_001~005.zip.part17179869184
├── Depth_001~005.zip.part18253611008
├── Depth_001~005.zip.part19327352832
├── Depth_001~005.zip.part2147483648
├── Depth_001~005.zip.part3221225472
├── Depth_001~005.zip.part4294967296
├── Depth_001~005.zip.part5368709120
├── Depth_001~005.zip.part6442450944
├── Depth_001~005.zip.part7516192768
├── Depth_001~005.zip.part8589934592
└── Depth_001~005.zip.part9663676416


In [3]:
from pathlib import Path

base = Path("C:/potenup3/transdeep/download (1)/15.인도보행영상/뎁스프리딕션")
zip_name = "Depth_001~005.zip"
out_zip = base / zip_name
extract_dir = base / "Depth_001~005_extracted"  # 압축 풀 폴더(원하면 이름 바꿔도 됨)

base, out_zip, extract_dir

(WindowsPath('C:/potenup3/transdeep/download (1)/15.인도보행영상/뎁스프리딕션'),
 WindowsPath('C:/potenup3/transdeep/download (1)/15.인도보행영상/뎁스프리딕션/Depth_001~005.zip'),
 WindowsPath('C:/potenup3/transdeep/download (1)/15.인도보행영상/뎁스프리딕션/Depth_001~005_extracted'))

In [4]:
parts = sorted(
    base.glob("Depth_001~005.zip.part*"),
    key=lambda p: int(p.name.split(".part")[-1])
)

print("parts:", len(parts))
assert len(parts) > 0, "part 파일을 못 찾음"

with open(out_zip, "wb") as w:
    for p in parts:
        with open(p, "rb") as r:
            w.write(r.read())

print("✅ merged zip:", out_zip)
print("zip size (MB):", out_zip.stat().st_size / 1024 / 1024)

parts: 19
✅ merged zip: C:\potenup3\transdeep\download (1)\15.인도보행영상\뎁스프리딕션\Depth_001~005.zip
zip size (MB): 19438.95959663391


In [2]:
from pathlib import Path

p = Path(r"C:/potenup3/transdeep/download (1)/15.인도보행영상/뎁스프리딕션/Depth_001~005_extracted/Depth_001/ZED1_KSC_001032_disp16.png")
print("exists:", p.exists())
print("size(bytes):", p.stat().st_size if p.exists() else None)
print("suffix:", p.suffix)

exists: True
size(bytes): 577904
suffix: .png


In [3]:
from PIL import Image
import numpy as np

img = Image.open(r"C:/potenup3/transdeep/download (1)/15.인도보행영상/뎁스프리딕션/Depth_001~005_extracted/Depth_001/ZED1_KSC_001032_disp16.png")
arr = np.array(img)

print(img.mode, arr.dtype, arr.shape)
print("min/max:", arr.min(), arr.max())

I;16 uint16 (1080, 1920)
min/max: 0 19609


In [4]:
from PIL import Image
import numpy as np

disp16_path = r"C:/potenup3/transdeep/download (1)/15.인도보행영상/뎁스프리딕션/Depth_001~005_extracted/Depth_001/ZED1_KSC_001032_disp16.png"
d16 = np.array(Image.open(disp16_path))  # uint16, (1080,1920)

H, W = d16.shape
samples = [
    (W//2, int(H*0.85)),  # 바닥쪽 중앙
    (W//2, int(H*0.60)),  # 중간
    (int(W*0.30), int(H*0.70)), # 왼쪽(전봇대 근처) 추정
]

for x,y in samples:
    print((x,y), int(d16[y,x]))

(960, 918) 0
(960, 648) 493
(576, 756) 3978


In [7]:
import numpy as np
from PIL import Image
import configparser

# ===== 0) 경로 =====
base = r"C:/potenup3/transdeep/download (1)/15.인도보행영상/뎁스프리딕션/Depth_001~005_extracted/Depth_001"
disp16_path = base + r"/ZED1_KSC_001032_disp16.png"
conf_path   = base + r"/ZED1_KSC_001032_confidence.png"
conf_ini    = base + r"/Depth_001.conf"   # (폴더 위치 다르면 수정)

# ===== 1) disp16, confidence 읽기 =====
disp16 = np.array(Image.open(disp16_path)).astype(np.float32)  # (H,W), uint16 -> float
conf   = np.array(Image.open(conf_path)).astype(np.float32)    # (H,W)

H, W = disp16.shape
print("disp16:", disp16.dtype, disp16.shape, disp16.min(), disp16.max())
print("conf:", conf.dtype, conf.shape, conf.min(), conf.max())

# ===== 2) 캘리브레이션(FHD) 읽기 =====
cfg = configparser.ConfigParser()
cfg.read(conf_ini, encoding="utf-8")

fx = float(cfg["LEFT_CAM_FHD"]["fx"])
fy = float(cfg["LEFT_CAM_FHD"]["fy"])
cx = float(cfg["LEFT_CAM_FHD"]["cx"])
cy = float(cfg["LEFT_CAM_FHD"]["cy"])
B  = float(cfg["STEREO"]["BaseLine"])  # mm로 가정

fB = fx * B
print("fx,fy,cx,cy,B:", fx,fy,cx,cy,B)

# ===== 3) disparity -> depth(mm) =====
SCALE = 16.0
disp = disp16 / SCALE  # disparity(px)

# 유효 마스크: disparity>0 + confidence로 필터
valid = disp > 0

# confidence가 보통 "밝을수록 신뢰 높음"이면 아래처럼 (상황에 따라 조절)
# conf가 0~255면 threshold 50~150 사이로 조절해봐
valid &= (conf > 50)

Z = np.zeros_like(disp, dtype=np.float32)
Z[valid] = fB / disp[valid]  # mm

print("Z(mm) valid range:", float(Z[valid].min()) if valid.any() else None, float(Z[valid].max()) if valid.any() else None)

# ===== 4) 바닥 ROI에서 3D 점 만들기 (간단 버전) =====
# 화면 아래 30%를 바닥 후보로 가정 (나중에 segmentation 있으면 더 좋음)
y0 = int(H * 0.70)
roi = np.zeros_like(valid)
roi[y0:H, :] = True

mask = valid & roi

ys, xs = np.where(mask)
Zs = Z[ys, xs]

# 3D 복원 (카메라 좌표)
Xs = (xs - cx) * (Zs / fx)
Ys = (ys - cy) * (Zs / fy)

pts = np.stack([Xs, Ys, Zs], axis=1)
print("points:", pts.shape)

# ===== 5) 평면 피팅 (최소제곱, RANSAC 없이 간단 버전) =====
# plane: ax + by + cz + d = 0
# [x y z 1] @ [a b c d] = 0  (SVD로 해)
A = np.c_[pts, np.ones(len(pts))]
_, _, Vt = np.linalg.svd(A, full_matrices=False)
plane = Vt[-1, :]  # [a,b,c,d]
a,b,c,d = plane

# 법선 벡터
n = np.array([a,b,c], dtype=np.float32)
n = n / (np.linalg.norm(n) + 1e-9)

# ===== 6) slope(각도) 계산 =====
# 카메라 좌표에서 "위쪽(up)"을 (0, -1, 0) 또는 (0,1,0) 중 하나로 두고
# 어느 쪽이든 각도는 abs로 동일하게 처리 가능
up = np.array([0, 1, 0], dtype=np.float32)
cos_theta = np.clip(np.dot(n, up), -1.0, 1.0)
theta = np.degrees(np.arccos(abs(cos_theta)))  # 0이면 수평, 커질수록 경사

print("Estimated slope(deg):", float(theta))
print("plane:", plane)

disp16: float32 (1080, 1920) 0.0 19609.0
conf: float32 (1080, 1920) 0.0 255.0
fx,fy,cx,cy,B: 1394.83 1394.83 932.377 559.96 120.009
Z(mm) valid range: 209.60044860839844 9430.5439453125
points: (82382, 3)
Estimated slope(deg): 7.852479457855225
plane: [ 6.00819831e-04  4.69408385e-03  2.41083789e-04 -9.99988773e-01]
